In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.patches import Rectangle
import matplotlib.patheffects as path_effects
import matplotlib.transforms as mtransforms
from matplotlib.tri import Triangulation

import os
from copy import copy

import analysis_tools as tool
from config import *

# Load Data

In [ ]:
grid_data = tool.load_grid_data()

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

exp_name = "REA"
da_rea_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_rea_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_CTL"
da_ctl_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_ctl_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_WLT"
da_wlt_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_wlt_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_SAT"
da_sat_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_sat_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

# Precipitation Maps

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T00"))
timeframe = int((time_slice.stop - time_slice.start) / np.timedelta64(1, 'h'))

### Deterministic

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_26"], da_rea_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_26_red"], da_sat_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="WET")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_26_red"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_26_red"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_26_red"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_26_red"]["clat"]) <= PLOT_WINDOW[3]))

diff = da_sat_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="WET - CTL")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_26_red"], da_ctl_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="CTL")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_26_red"], da_wlt_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="DRY")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = da_wlt_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="DRY - CTL")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.07, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color="white", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="black")])
    
plt.savefig(f'./figs/figure_03.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_28"], ds_rea_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_28"], ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_28"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_28"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_28"]["clat"]) <= PLOT_WINDOW[3]))

diff = ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_28"], ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_28"], ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.savefig(f'./figs/map_comp_ens.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

# Time Series

## Precipitation

### Deterministic

In [ ]:
# Compute time series of deterministic runs: 
cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
ts_rea = (da_rea_tp * grid_data["area_26"]).sel(cell=grid_data["focus_cells_26"]).sum(dim="cell")
ts_ctl = (da_ctl_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_wlt = (da_wlt_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_sat = (da_sat_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")
ax.plot(ts_ctl["valid_time"], ts_ctl, color="tab:blue", label="CTL")
ax.plot(ts_wlt["valid_time"], ts_wlt, color="tab:green", label="WLT")
ax.plot(ts_sat["valid_time"], ts_sat, color="tab:orange", label="SAT")

plt.legend()

ax.set(ylabel="Area precipitation in kg/h")

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_deterministic.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
# Compute time series of ensemble runs:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_rea_ens = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"]).sum(dim="cell")
ts_ctl_ens = (ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_wlt_ens = (ds_wlt_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_sat_ens = (ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color=c_dry, label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color=c_wet, label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color=c_rea, label="REA")

#plt.legend()

ax.set(ylabel="Area precipitation in kg/h")
ax.text(0.03, 0.96, "a", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_ensemble.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
ctl_allsum = ts_ctl_ens.median(dim="mem").sum().item()
rea_allsum = ts_rea_ens.median(dim="mem").sum().item()
wlt_allsum = ts_wlt_ens.median(dim="mem").sum().item()
sat_allsum = ts_sat_ens.median(dim="mem").sum().item()

print(f"CTL (median) produces {(ctl_allsum - rea_allsum) / rea_allsum * 100:.02f}% more/less precipitation than DREAM (median).")
print(f"SAT (median) produces {(sat_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")
print(f"WLT (median) produces {(wlt_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")

## Soil Moisture

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

mask = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PRUDENCE_REGIONS["ME"]["lon"].start) & 
        (np.rad2deg(grid_data["grid_28"]["clon"]) >= PRUDENCE_REGIONS["ME"]["lon"].stop) & 
        (np.rad2deg(grid_data["grid_28"]["clat"]) >= PRUDENCE_REGIONS["ME"]["lat"].start) & 
        (np.rad2deg(grid_data["grid_28"]["clat"]) >= PRUDENCE_REGIONS["ME"]["lat"].stop))

In [ ]:
def compute_wso_iqr(exp_name, base_dir, dts, mask):
    """Just a helper function to limit memory usage."""
    ds_ens_wso = tool.read_merged_var_ens("W_SO", dts, f"{base_dir}/{exp_name}/merged/W_SO", accu=False)
    _ = ds_ens_wso.isel(cell=mask, depthBelowLandLayer=0).mean(dim="cell")
    
    pct_25 = _.quantile(0.25, dim="mem")
    pct_50 = _.quantile(0.5, dim="mem")
    pct_75 = _.quantile(0.75, dim="mem")

    return pct_25, pct_50, pct_75

rea_25, rea_50, rea_75 = compute_wso_iqr("REA", base_dir, dts, mask)
ctl_25, ctl_50, ctl_75 = compute_wso_iqr("BLK_CTL", base_dir, dts, mask)
wlt_25, wlt_50, wlt_75 = compute_wso_iqr("BLK_WLT", base_dir, dts, mask)
sat_25, sat_50, sat_75 = compute_wso_iqr("BLK_SAT", base_dir, dts, mask)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

plt.legend()

ax.set(ylabel=r"Average soil moisture in kg/m$^2$")
ax.text(0.03, 0.96, "b", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_soil_moisture.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
ctl_smm = ctl_50.median().item()
rea_smm = rea_50.median().item()
wlt_smm = wlt_50.median().item()
sat_smm = sat_50.median().item()

print(f"CTL (median) has {(ctl_smm - rea_smm) / rea_smm * 100:.02f}% more/less top level soil moisture than DREAM (median).")
print(f"SAT (median) has {(sat_smm - ctl_smm) / ctl_smm * 100:.02f}% more/less top level soil moisture than CTL (median).")
print(f"WLT (median) has {(wlt_smm - ctl_smm) / ctl_smm * 100:.02f}% more/less top level soil moisture than CTL (median).")

## Precip & SM in one plot

In [ ]:
fig, axs = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
fig.tight_layout(h_pad=0.8)

# Precip
ax = axs[0]
ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color="tab:blue", label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color="tab:blue", alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color="tab:green", label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color="tab:green", alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color="tab:orange", label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color="tab:orange", alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color="black", label="REA")

ax.legend()

ax.set(ylabel=r"Area precipitation in kg h$^{-1}$")
ax.text(0.02, 0.96, "(a)", transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# SM
ax = axs[1]
ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

#plt.legend()

ax.set(ylabel=r"Average soil moisture in kg m$^{-2}$")
ax.text(0.02, 0.9, "(b)", transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_04.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## Evaporation

In [ ]:
R = 6371000  # Earth radius (m)
lat_spacing = 0.15
lon_spacing = 0.15

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("data/moisture_tracking/REA/expanded_region.nc").rename({"latitude": "lat", "longitude": "lon"})

lat_rad = np.deg2rad(ll_grid["lat"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_mt = area.reindex({"lat": area["lat"][::-1]}) + 0 * ll_grid["lon"] #hack to broadcast to lat/lon

fr_land_mt = xr.open_dataset("data/moisture_tracking/invar/fr_land_015x015_EUL.nc")["FR_LAND"]
fr_land_mt = fr_land_mt.reindex({"lat": fr_land_mt["lat"][::-1]}).squeeze()

In [ ]:
def read_mt_output(exp_name, var):
    """
    Helper function to declutter the renaming and reindexing operations for reading tracking output data.
    """
    da_track = xr.open_mfdataset(f"data/moisture_tracking/{exp_name}/output/det/backtrack_2021-07-??T00-00.nc")[var]
    da_track = da_track.rename({"latitude": "lat", "longitude": "lon"})
    da_track = da_track.reindex({"lat": da_track["lat"][::-1]})
    return da_track

In [ ]:
da_rea_track = read_mt_output("REA", "e_track")
da_ctl_track = read_mt_output("BLK_CTL", "e_track")
da_sat_track = read_mt_output("BLK_SAT", "e_track")
da_wlt_track = read_mt_output("BLK_WLT", "e_track")

In [ ]:
mask_evapt = fr_land_mt.where(fr_land_mt > 0.8)
shares_track = (da_rea_track + da_ctl_track + da_wlt_track + da_sat_track).sum(dim="time") / (da_rea_track + da_ctl_track + da_wlt_track + da_sat_track).sum()
weights_evapt = area_mt.values * shares_track #minor coordinate misalignment in area compared to shares

In [ ]:
da_rea_evapt = xr.open_mfdataset("data/moisture_tracking/REA/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_rea_evapt = da_rea_evapt.reindex({"lat": da_rea_evapt["lat"][::-1]})
da_rea_evapt *= -weights_evapt.values

da_ctl_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_CTL/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_ctl_evapt = da_ctl_evapt.reindex({"lat": da_ctl_evapt["lat"][::-1]})
da_ctl_evapt *= -weights_evapt.values

da_sat_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_SAT/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_sat_evapt = da_sat_evapt.reindex({"lat": da_sat_evapt["lat"][::-1]})
da_sat_evapt *= -weights_evapt.values

da_wlt_evapt = xr.open_mfdataset("data/moisture_tracking/BLK_WLT/det/icon_R03B07_e_202107??.nc")["EVAPT"]
da_wlt_evapt = da_wlt_evapt.reindex({"lat": da_wlt_evapt["lat"][::-1]})
da_wlt_evapt *= -weights_evapt.values

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(da_ctl_evapt["time"], da_ctl_evapt.mean(dim=["lat", "lon"]), color=c_ctl, label="CTL")

ax.plot(da_wlt_evapt["time"], da_wlt_evapt.mean(dim=["lat", "lon"]), color=c_dry, label="Dry")

ax.plot(da_sat_evapt["time"], da_sat_evapt.mean(dim=["lat", "lon"]), color=c_wet, label="Wet")

ax.plot(da_rea_evapt["time"], da_rea_evapt.mean(dim=["lat", "lon"]), color=c_rea, label="REA")

plt.legend()

ax.set(ylabel=r"Weighted average evapotranspiration in kg s$^{-1}$")

plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_06.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

## CAPE

In [ ]:
ds_cape_ctl = xr.open_mfdataset("data/BLK_CTL/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_ctl_ens = xr.concat([xr.open_mfdataset(f"data/BLK_CTL/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

ds_cape_wlt = xr.open_mfdataset("data/BLK_WLT/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_wlt_ens = xr.concat([xr.open_mfdataset(f"data/BLK_WLT/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

ds_cape_sat = xr.open_mfdataset("data/BLK_SAT/merged/CAPE/fc_R03B07_cape_merged.202107????")["cape"]
ds_cape_sat_ens = xr.concat([xr.open_mfdataset(f"data/BLK_SAT/merged/CAPE/fc_R02B07_N02_cape_merged.202107????.{mmm:03}") for mmm in range(1,21)], dim="mem")["cape"]

In [ ]:
lons, lats = np.rad2deg(grid_data["grid_26_red"]["clon"]), np.rad2deg(grid_data["grid_26_red"]["clat"])
mask = (lons > PASSIVE_REGION["x0"]) & (lons < PASSIVE_REGION["x1"]) & (lats > PASSIVE_REGION["y0"]) & (lats < PASSIVE_REGION["y1"])

tri_passive = Triangulation(lons[mask].values, lats[mask].values)

mask_cape_26 = ((lons[mask] > FOCUS_REGION["x0"]) 
              & (lons[mask] < FOCUS_REGION["x1"]) 
              & (lats[mask] > FOCUS_REGION["y0"])
              & (lats[mask] < FOCUS_REGION["y1"]))

lons, lats = np.rad2deg(grid_data["grid_28"]["clon"]), np.rad2deg(grid_data["grid_28"]["clat"])
mask = (lons > PASSIVE_REGION["x0"]) & (lons < PASSIVE_REGION["x1"]) & (lats > PASSIVE_REGION["y0"]) & (lats < PASSIVE_REGION["y1"])
mask_cape_28 = ((lons[mask] > FOCUS_REGION["x0"]) 
              & (lons[mask] < FOCUS_REGION["x1"]) 
              & (lats[mask] > FOCUS_REGION["y0"])
              & (lats[mask] < FOCUS_REGION["y1"]))

In [ ]:
fig, ax = plt.subplots()

ax.plot(ds_cape_ctl["time"], ds_cape_ctl.sel(cell=mask_cape_26).mean(dim="cell"), label="CTL")
ax.plot(ds_cape_wlt["time"], ds_cape_wlt.sel(cell=mask_cape_26).mean(dim="cell"), label="WLT")
ax.plot(ds_cape_sat["time"], ds_cape_sat.sel(cell=mask_cape_26).mean(dim="cell"), label="SAT")

plt.legend()
plt.xticks(rotation=45)
ax.set(ylabel="CAPE in J/kg")

plt.show()

## TP Det+IQR & CP Det+IQR & CAPE & SM & Evap

In [ ]:
def get_cp_ts(dts, base_dir):
    """Helper function to limit memory usage."""
    exp_name = "BLK_CTL"
    da_ctl_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_ctl_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    exp_name = "BLK_WLT"
    da_wlt_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_wlt_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    exp_name = "BLK_SAT"
    da_sat_cp = tool.read_merged_var_det("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)
    ds_sat_ens_cp = tool.read_merged_var_ens("cp", dts, f"{base_dir}/{exp_name}/merged/cp", accu=True)

    # Compute time series of deterministic runs: 
    cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
    ts_ctl_cp = (da_ctl_cp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
    ts_wlt_cp = (da_wlt_cp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
    ts_sat_cp = (da_sat_cp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")

    # Compute time series of ensemble runs:
    cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
    ts_ctl_ens_cp = (ds_ctl_ens_cp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
    ts_wlt_ens_cp = (ds_wlt_ens_cp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
    ts_sat_ens_cp = (ds_sat_ens_cp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")

    return( ts_ctl_cp, ts_wlt_cp, ts_sat_cp, ts_ctl_ens_cp, ts_wlt_ens_cp, ts_sat_ens_cp )

dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")
base_dir = "/automount/agh/s6tifohr/july21_eval/data"
ts_ctl_cp, ts_wlt_cp, ts_sat_cp, ts_ctl_ens_cp, ts_wlt_ens_cp, ts_sat_ens_cp = get_cp_ts(dts, base_dir)

In [ ]:
fig, axs = plt.subplots(5,1, figsize=(8,10), sharex=True, gridspec_kw={'height_ratios': [2, 2*4/9, 1, 1, 1]})
fig.tight_layout(h_pad=0.8)
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans)

# TP
ax = axs[0]
ax.plot(ts_ctl["valid_time"], ts_ctl, color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt["valid_time"], ts_wlt, color=c_dry, label="DRY")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat["valid_time"], ts_sat, color=c_wet, label="WET")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")

ax.legend()
ax.set_ylim(0, 7e11)
ax.set(ylabel=r"$\sum$ TP in kg h$^{-1}$")
ax.text(0, 1, "(a)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# CP
ax = axs[1]
ax.plot(ts_ctl_cp["valid_time"], ts_ctl_cp, color=c_ctl, label="CTL")
ax.fill_between(ts_ctl_ens_cp["valid_time"], ts_ctl_ens_cp.quantile(0.25, dim="mem"), ts_ctl_ens_cp.quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ts_wlt_cp["valid_time"], ts_wlt_cp, color=c_dry, label="DRY")
ax.fill_between(ts_wlt_ens_cp["valid_time"], ts_wlt_ens_cp.quantile(0.25, dim="mem"), ts_wlt_ens_cp.quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ts_sat_cp["valid_time"], ts_sat_cp, color=c_wet, label="WET")
ax.fill_between(ts_sat_ens_cp["valid_time"], ts_sat_ens_cp.quantile(0.25, dim="mem"), ts_sat_ens_cp.quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.set_ylim(0, 3e11)
ax.set(ylabel=r"$\sum$ CP in kg h$^{-1}$")
ax.text(0, 1, "(b)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# CAPE
ax = axs[2]
ax.plot(ds_cape_ctl["time"], ds_cape_ctl.sel(cell=mask_cape_26).mean(dim="cell"), color=c_ctl, label="CTL")
ax.fill_between(ds_cape_ctl_ens["time"], 
                ds_cape_ctl_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_ctl_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_ctl, alpha=0.3)

ax.plot(ds_cape_wlt["time"], ds_cape_wlt.sel(cell=mask_cape_26).mean(dim="cell"), color=c_dry, label="DRY")
ax.fill_between(ds_cape_wlt_ens["time"], 
                ds_cape_wlt_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_wlt_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_dry, alpha=0.3)

ax.plot(ds_cape_sat["time"], ds_cape_sat.sel(cell=mask_cape_26).mean(dim="cell"), color=c_wet, label="WET")
ax.fill_between(ds_cape_sat_ens["time"], 
                ds_cape_sat_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.25, dim="mem"), 
                ds_cape_sat_ens.sel(cell=mask_cape_28).mean(dim="cell").chunk(dict(mem=-1)).quantile(0.75, dim="mem"), color=c_wet, alpha=0.3)

ax.set(ylabel=r"$\langle$ CAPE $\rangle$ in J kg$^{-1}$")
ax.text(0, 1, "(c)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# SM
ax = axs[3]
ax.plot(ctl_50["valid_time"], ctl_50, color=c_ctl, label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color=c_ctl, alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color=c_dry, label="DRY")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color=c_dry, alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color=c_wet, label="WET")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color=c_wet, alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color=c_rea, label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color=c_rea, alpha=0.3)

ax.set(ylabel=r"$\langle$ SM $\rangle$ in kg m$^{-2}$")
ax.text(0, 1, "(d)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


# EVAPT
ax = axs[4]
ax.plot(da_ctl_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_ctl_evapt.sel(time=slice(dts[0], dts[-1])).mean(dim=["lat", "lon"]), color=c_ctl, label="CTL")
ax.plot(da_wlt_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_wlt_evapt.sel(time=slice(dts[0], dts[-1])).mean(dim=["lat", "lon"]), color=c_dry, label="Dry")
ax.plot(da_sat_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_sat_evapt.sel(time=slice(dts[0], dts[-1])).mean(dim=["lat", "lon"]), color=c_wet, label="Wet")
ax.plot(da_rea_evapt.sel(time=slice(dts[0], dts[-1]))["time"], da_rea_evapt.sel(time=slice(dts[0], dts[-1])).mean(dim=["lat", "lon"]), color=c_rea, label="REA")

ax.set(ylabel=r"$\langle$ EVAPT $\rangle_\text{w}$ in kg s$^{-1}$")
ax.text(0, 1, "(e)", transform=ax.transAxes + trans,
        ha="left", va="top",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.xticks(rotation=30)
plt.savefig(f'./figs/figure_04.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Scatter Plot

In [ ]:
dts = pd.date_range("2021-07-13T00", "2021-07-15T00", freq="3h")

# PRMSL
#ds_prmsl_rea = tool.read_merged_var_det("prmsl", dts, "data/REA/merged/prmsl", accu=False) / 100.

ds_prmsl_ctl = tool.read_merged_var_det("prmsl", dts, "data/BLK_CTL/merged/prmsl", accu=False) / 100.
ds_prmsl_ctl_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_CTL/merged/prmsl", accu=False) / 100.

ds_prmsl_sat = tool.read_merged_var_det("prmsl", dts, "data/BLK_SAT/merged/prmsl", accu=False) / 100.
ds_prmsl_sat_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_SAT/merged/prmsl", accu=False) / 100.

ds_prmsl_wlt = tool.read_merged_var_det("prmsl", dts, "data/BLK_WLT/merged/prmsl", accu=False) / 100.
ds_prmsl_wlt_ens = tool.read_merged_var_ens("prmsl", dts, "data/BLK_WLT/merged/prmsl", accu=False) / 100.


# TOT_PRECIP
#ds_tp_rea = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/REA/merged/tp", accu=True)

ds_tp_ctl = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_CTL/merged/tp", accu=True)
ds_tp_ctl_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_CTL/merged/tp", accu=True)

ds_tp_wlt = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_WLT/merged/tp", accu=True)
ds_tp_wlt_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_WLT/merged/tp", accu=True)

ds_tp_sat = tool.read_merged_var_det("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_SAT/merged/tp", accu=True)
ds_tp_sat_ens = tool.read_merged_var_ens("tp", dts, "/automount/agh/s6tifohr/july21_eval/data/BLK_SAT/merged/tp", accu=True)


# GPH
ds_gph_ctl = tool.read_merged_var_det("gph", dts, "data/BLK_CTL/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_ctl_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_CTL/merged/gph", accu=False, format="netcdf").sel(plev=50000)

ds_gph_sat = tool.read_merged_var_det("gph", dts, "data/BLK_SAT/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_sat_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_SAT/merged/gph", accu=False, format="netcdf").sel(plev=50000)

ds_gph_wlt = tool.read_merged_var_det("gph", dts, "data/BLK_WLT/merged/gph", accu=False, format="netcdf").sel(plev=50000)
ds_gph_wlt_ens = tool.read_merged_var_ens("gph", dts, "data/BLK_WLT/merged/gph", accu=False, format="netcdf").sel(plev=50000)

In [ ]:
# Compute precip sums DET:
cell_areas_focus_det = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
lats, lons = np.rad2deg(grid_data["grid_26_red"]["clat"].values), np.rad2deg(grid_data["grid_26_red"]["clon"].values)
mask_mean_26_red = (lons > FOCUS_REGION_L["x0"]) & (lons < FOCUS_REGION_L["x1"]) & (lats > FOCUS_REGION_L["y0"]) & (lats < FOCUS_REGION_L["y1"])

#sums_rea_det = (ds_tp_rea.sel(cell=grid_data["focus_cells_26"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_ctl_det = (ds_tp_ctl.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_sat_det = (ds_tp_sat.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_wlt_det = (ds_tp_wlt.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])


# Compute precip sums ENS:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
lats, lons = np.rad2deg(grid_data["grid_28"]["clat"].values), np.rad2deg(grid_data["grid_28"]["clon"].values)
mask_mean_28 = (lons > FOCUS_REGION_L["x0"]) & (lons < FOCUS_REGION_L["x1"]) & (lats > FOCUS_REGION_L["y0"]) & (lats < FOCUS_REGION_L["y1"])

sums_ctl_ens = (ds_tp_ctl_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])
sums_sat_ens = (ds_tp_sat_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])
sums_wlt_ens = (ds_tp_wlt_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])

In [ ]:
# Compute precip sums DET:
cell_areas_focus_det = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
lats, lons = np.rad2deg(grid_data["grid_26_red"]["clat"].values), np.rad2deg(grid_data["grid_26_red"]["clon"].values)
mask_mean_26_red = (lons > FOCUS_REGION["x0"]-2) & (lons < FOCUS_REGION["x1"]+2) & (lats > FOCUS_REGION["y0"]-2) & (lats < FOCUS_REGION["y1"]+2)

#sums_rea_det = (ds_tp_rea.sel(cell=grid_data["focus_cells_26"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_ctl_det = (ds_tp_ctl.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_sat_det = (ds_tp_sat.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])
sums_wlt_det = (ds_tp_wlt.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus_det).sum(dim=["cell", "step"])


# Compute precip sums ENS:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
lats, lons = np.rad2deg(grid_data["grid_28"]["clat"].values), np.rad2deg(grid_data["grid_28"]["clon"].values)
mask_mean_28 = (lons > FOCUS_REGION["x0"]-2) & (lons < FOCUS_REGION["x1"]+2) & (lats > FOCUS_REGION["y0"]-2) & (lats < FOCUS_REGION["y1"]+2)

sums_ctl_ens = (ds_tp_ctl_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])
sums_sat_ens = (ds_tp_sat_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])
sums_wlt_ens = (ds_tp_wlt_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim=["cell", "step"])

In [ ]:
from scipy.stats import linregress

def bootstrap_ci(x, y, x_pred, n_b, level):
    """
    Performs a linear fit to `x` and `y`, returning the fitted values corresponding to the
    evaluation points `x_pred` and the r value. The function also returns the bootstrapped confidence 
    intervals at `x_pred` to a confidence level of `level` using `n_b` bootstrap samples. 
    """
    reg_res = linregress(x, y)
    y_pred = reg_res.slope * x_pred + reg_res.intercept
    
    slopes, intercepts = np.full(n_b, np.nan), np.full(n_b, np.nan)

    for i in range(n_b):
        ii = np.random.randint(low=0, high=20, size=20)
        reg_res = linregress(x[ii], y[ii])
        slopes[i], intercepts[i] = reg_res.slope, reg_res.intercept

    y_boot = np.array([intercepts + slopes * xp for xp in x_pred])

    ci_low, ci_high = np.quantile(y_boot, (1-level)/2, axis=1), np.quantile(y_boot, 1-(1-level)/2, axis=1)

    return y_pred, linregress(x, y).rvalue, ci_low, ci_high

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,3), sharex=True, sharey=True)
fig.tight_layout()
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently


# CONTROL
ax = axs[0]

x, y = ds_prmsl_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl, s=25)
#for mem_id, xi, yi in zip(x.mem.values, x.values, y.values):
#    ax.annotate(str(mem_id), (xi, yi), fontsize=5, ha="center", va="center", color="white")

# Compute fit and confidence intervals...
x_pred = np.arange(990, 1009)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_ctl, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_ctl, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)


ax.scatter(ds_prmsl_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="CTL", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa", ylabel=r"$\sum$ TP in kg")
ax.text(0, 1, "(a)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# WET
ax = axs[1]

x, y = ds_prmsl_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet, s=25)

x_pred = np.arange(984, 1007)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

ax.plot(x_pred, y_pred, color=c_wet, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_wet, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_prmsl_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="WET", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa")
ax.text(0, 1, "(b)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# DRY
ax = axs[2]

x, y = ds_prmsl_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry, s=25)

# Compute fit and confidence intervals...
x_pred = np.arange(993, 1008)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_dry, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_dry, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)


ax.scatter(ds_prmsl_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="DRY", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa")
ax.text(0, 1, "(c)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.savefig(f'./figs/figure_scatter.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,3), sharex=True, sharey=True)
fig.tight_layout()
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently


# CONTROL
ax = axs[0]

x, y = ds_gph_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl)

# Compute fit and confidence intervals...
x_pred = np.arange(5450, 5700, 10)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_ctl, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_ctl, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="Control", xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m", ylabel=r"$\sum$ TP in kg")
ax.text(0, 1, "(d)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# WET
ax = axs[1]
x, y = ds_gph_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet)

#x_pred = np.arange(984, 1007)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

ax.plot(x_pred, y_pred, color=c_wet, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_wet, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="Wet", xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m")
ax.text(0, 1, "(e)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# DRY
ax = axs[2]

x, y = ds_gph_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry)

# Compute fit and confidence intervals...
#x_pred = np.arange(993, 1008)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_dry, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_dry, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="Dry", xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m")
ax.text(0, 1, "(f)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])


plt.savefig(f'./figs/figure_scatter_gph.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,6), sharey=True)
fig.tight_layout(h_pad=2)
trans = mtransforms.ScaledTranslation(6/72, -10/72, fig.dpi_scale_trans) #to position text consistently


# CONTROL - PMSL
ax = axs[0,0]
ax.set_xlim(983, 1007)

x, y = ds_prmsl_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl, s=25)
#for mem_id, xi, yi in zip(x.mem.values, x.values, y.values):
#    ax.annotate(str(mem_id), (xi, yi), fontsize=5, ha="center", va="center", color="white")

# Compute fit and confidence intervals...
x_pred = np.arange(990, 1009)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_ctl, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_ctl, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)


ax.scatter(ds_prmsl_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="CTL", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa", ylabel=r"$\sum$ TP in kg")
ax.text(0, 1, "(a)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# WET - PMSL
ax = axs[0,1]
ax.set_xlim(983, 1007)

x, y = ds_prmsl_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet, s=25)

x_pred = np.arange(984, 1007)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

ax.plot(x_pred, y_pred, color=c_wet, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_wet, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_prmsl_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="WET", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa")
ax.text(0, 1, "(b)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# DRY - PMSL
ax = axs[0,2]
ax.set_xlim(983, 1007)

x, y = ds_prmsl_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry, s=25)

# Compute fit and confidence intervals...
x_pred = np.arange(993, 1007)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_dry, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_dry, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)


ax.scatter(ds_prmsl_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X")

# labels
ax.set(title="DRY", xlabel=r"Min[p$_\text{msl}$]$_\text{A,t}$ in hPa")
ax.text(0, 1, "(c)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# CONTROL - GPH
ax = axs[1,0]
ax.set_xlim(5460, 5690)

x, y = ds_gph_ctl_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_ctl_ens
ax.scatter(x, y, color=c_ctl)

# Compute fit and confidence intervals...
x_pred = np.arange(5510, 5670, 10)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_ctl, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_ctl, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_ctl.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_ctl_det, color=c_ctl, edgecolors="white", s=70, marker="X")

# labels
ax.set(xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m", ylabel=r"$\sum$ TP in kg")
ax.text(0, 1, "(d)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# WET - GPH
ax = axs[1,1]
ax.set_xlim(5460, 5690)

x, y = ds_gph_sat_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_sat_ens
ax.scatter(x, y, color=c_wet)

x_pred = np.arange(5480, 5690, 10)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

ax.plot(x_pred, y_pred, color=c_wet, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_wet, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_sat.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_sat_det, color=c_wet, edgecolors="white", s=70, marker="X")

# labels
ax.set(xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m")
ax.text(0, 1, "(e)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



# DRY - GPH
ax = axs[1,2]
ax.set_xlim(5460, 5690)

x, y = ds_gph_wlt_ens.sel(cell=mask_mean_28).min(dim=("cell", "step")), sums_wlt_ens
ax.scatter(x, y, color=c_dry)

# Compute fit and confidence intervals...
x_pred = np.arange(5540, 5670, 10)
y_pred, r, ci_low, ci_high = bootstrap_ci(x, y, x_pred, 10000, 0.95)

# ...and plot them
ax.plot(x_pred, y_pred, color=c_dry, linestyle="dashed", zorder=-1)
ax.fill_between(x_pred, ci_low, ci_high, color=c_dry, alpha=0.15)
ax.text(0.9, 1, r"R$^2 = $" + f"{r**2:.3f}", transform=ax.transAxes + trans,
        ha="right", va="top", color="black", zorder=1,)

ax.scatter(ds_gph_wlt.sel(cell=mask_mean_26_red).min(dim=("cell", "step")), sums_wlt_det, color=c_dry, edgecolors="white", s=70, marker="X")

# labels
ax.set(xlabel=r"Min[h$_\text{500}$]$_\text{A,t}$ in m")
ax.text(0, 1, "(f)", transform=ax.transAxes + trans,
        ha="left", va="top", color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")])



plt.savefig(f'./figs/figure_05.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Assimilation Region Showcase

In [ ]:
# This is not actually coinciding with the passive region, which is -10 to 40E and 30 to 75N, 
# but I used it to get a background in the image below  
mask_passive_region = ((np.rad2deg(grid_data["grid_26"]["clon"]) >= -20) & (np.rad2deg(grid_data["grid_26"]["clon"]) <= 50) & 
                       (np.rad2deg(grid_data["grid_26"]["clat"]) >= 20) & (np.rad2deg(grid_data["grid_26"]["clat"]) <= 85)).values
lons_pr, lats_pr = np.rad2deg(grid_data["grid_26"]["clon"][mask_passive_region]), np.rad2deg(grid_data["grid_26"]["clat"][mask_passive_region])

ii_passive_region = np.arange(len(mask_passive_region))[mask_passive_region]

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([-30, 70, 25, 80], crs=ccrs.PlateCarree())

gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl1.top_labels = False
gl1.right_labels = False

im = ax.tricontourf(grid_data["tri_28"], np.zeros(len(grid_data["area_28"])))
ax.text(50, 32, "EU Nest", color="white", weight="bold")

# Create a rectangle patch for the passive observation region:
box_passive = Rectangle((PASSIVE_REGION["x0"], PASSIVE_REGION["y0"]), PASSIVE_REGION["wx"], PASSIVE_REGION["wy"], edgecolor="purple", linewidth=2, fill=False)
ax.add_patch(box_passive)
ax.text(-9, 72, "Passive Observations", color="purple", weight="bold")

# Create a rectangle patch for the focus region
box_focus = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor="red", linewidth=2, fill=False)
ax.add_patch(box_focus)
ax.text(11, 50, "Focus Region", color="red", weight="bold")

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)

plt.savefig("figs/figure_01.png", dpi=300, bbox_inches='tight', format='png')
plt.show()

# Predictability

Get cell areas on lat-lon grid:

In [ ]:
R = 6371000  # Earth radius (m)
lat_spacing = 0.2
lon_spacing = 0.2

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("./data/free_forecasts/2021071200/mem001/fc_ll_DOM02_0001.nc")

lat_rad = np.deg2rad(ll_grid["lat"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_fc = area + 0 * ll_grid["lon"] #hack to broadcast to lat/lon

Need mid-points of the focus region:

In [ ]:
mx = (FOCUS_REGION["x0"] + FOCUS_REGION["x1"])/2
my = (FOCUS_REGION["y0"] + FOCUS_REGION["y1"])/2
mt = np.datetime64("2021-07-14T00")

Precipitation sums in free forecasts

In [ ]:
init_sums = {} #sums for all initialization dates
dts_fc = pd.date_range("2021-07-01T00", "2021-07-12T00", freq="D")

for dt in dts_fc:
    print(dt)
    
    ens_sums = np.full(20, np.nan) #maximum sum for all members of one initialization date

    for mem in range(1,21):
        path = f"./data/free_forecasts/{dt.year}{dt.month:02}{dt.day:02}00/mem{mem:03}/"
        fnames = [path + fname for fname in sorted(os.listdir(path))[-97:]]
        da_mem = xr.open_mfdataset(fnames)["tot_prec"].diff(dim="time")        

        da_mem = da_mem.sel(lon=slice(FOCUS_REGION["x0"] - 3, FOCUS_REGION["x1"] + 3), 
                            lat=slice(FOCUS_REGION["y0"] - 3, FOCUS_REGION["y1"] + 3), 
                            time=slice(np.datetime64("2021-07-12"), np.datetime64("2021-07-16")), 
                            drop=True) * area_fc

        rolling_sum = da_mem.rolling(lon=int(FOCUS_REGION["wx"]/lon_spacing), lat=int(FOCUS_REGION["wy"]/lat_spacing), time=48, center=True).sum().compute()
        cutout = rolling_sum.sel(lon=slice(mx-1, mx+1), lat=slice(my-1, my+1), time=slice(mt-np.timedelta64(12,"h"), mt+np.timedelta64(12,"h")))

        # If the indeces are needed:
        #ii_max = cutout.argmax(..., skipna=True)
        #cutout_max = cutout.isel(time=ii_max["time"], lat=ii_max["lat"], lon=ii_max["lon"])
        #x0_max, y0_max = cutout_max["lon"] - wx/2, cutout_max["lat"] - wy/2

        ens_sums[mem-1] = cutout.max(skipna=True)

    init_sums[dt] = ens_sums

Precipitation sums in storyline scenarios:

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T00"))

rea_sums = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
ctl_sums = (ds_ctl_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
sat_sums = (ds_sat_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
wlt_sums = (ds_wlt_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])

Amount of moisture backtracked:

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

area_weights = np.cos(np.deg2rad(ds_track_rea["latitude"]))

sum_tagged = ds_track_rea["tagged_precip"].weighted(area_weights).sum().values
ts_evap = ds_track_rea["e_track"].weighted(area_weights).sum(dim=["latitude", "longitude"])
ts_evap = ts_evap[::-1].cumsum()[::-1]
ts_loss = ds_track_rea["losses"].weighted(area_weights).sum(dim=["latitude", "longitude"])

In [ ]:
fig, ax1 = plt.subplots()


# Plot share of evaporation:
ax1.plot(range(1,13), ts_evap[:12] / sum_tagged)
ax1.set_ylim(0,1)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set(ylabel="Share of precipitation tracked to source")

# x orientation:
ax1.tick_params(axis="x", labelrotation=45)


# Add explaination to FC init.
x1, x2 = 1, 12
y1, y2 = -0.15, -0.18

ax1.vlines([x1, x2], y1-0.02, y1+0.02, colors='k', lw=1.5,
          transform=ax1.get_xaxis_transform(), clip_on=False)
ax1.plot([x1, x2], [y1, y1], 'k-', lw=1.5, clip_on=False,
        transform=ax1.get_xaxis_transform())

ax1.text((x1+x2)/2, y2, "Forecast initialized on", ha="center", va="top",
        transform=ax1.get_xaxis_transform())


# Plot simulated precipitation sums:
ax2 = ax1.twinx()

x = [init_sums[dt] for dt in init_sums.keys()] + [rea_sums.values, ctl_sums.values]#, wlt_sums.values, sat_sums.values]
tick_labels = [f"{dt.day}.{dt.month}." for dt in dts_fc] + ["DREAM", "LDA CTL"]#, "LDA Dry", "LDA Wet"] #tick labels are overriden by boxplot

bp = ax2.boxplot(x, tick_labels=tick_labels)
ax2.set(ylabel="Area Precipitation in kg")
ax2.set_ylim(0, 2.1e13)

# Highlight ICON-DREAM:
idx, lw = -2, 1.5
bp["boxes"][idx].set_linewidth(lw)
bp["whiskers"][2*idx].set_linewidth(lw)
bp["whiskers"][2*idx+1].set_linewidth(lw)
bp["caps"][2*idx].set_linewidth(lw)
bp["caps"][2*idx+1].set_linewidth(lw)


plt.savefig(f'./figs/figure_02.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Moisture Tracking

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

fnames_ctl = [f"data/moisture_tracking/BLK_CTL/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_ctl = xr.open_mfdataset(fnames_ctl)
ds_track_ctl = ds_track_ctl.reindex(latitude=list(reversed(ds_track_ctl["latitude"])))

fnames_wlt = [f"data/moisture_tracking/BLK_WLT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_wlt = xr.open_mfdataset(fnames_wlt)
ds_track_wlt = ds_track_wlt.reindex(latitude=list(reversed(ds_track_wlt["latitude"])))

fnames_sat = [f"data/moisture_tracking/BLK_SAT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_sat = xr.open_mfdataset(fnames_sat)
ds_track_sat = ds_track_sat.reindex(latitude=list(reversed(ds_track_sat["latitude"])))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,5), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=1, h_pad=1)

ax = axs[0,0]
im = ax.contourf(ds_track_rea["longitude"], ds_track_rea["latitude"], ds_track_rea["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="ICON-DREAM")

ax = axs[0,1]
im = ax.contourf(ds_track_sat["longitude"], ds_track_sat["latitude"], ds_track_sat["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="WET")

ax = axs[1,0]
im = ax.contourf(ds_track_ctl["longitude"], ds_track_ctl["latitude"], ds_track_ctl["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="CTL")

ax = axs[1,1]
im = ax.contourf(ds_track_wlt["longitude"], ds_track_wlt["latitude"], ds_track_wlt["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="DRY")


for ax, char in zip(axs.reshape(-1), ["(a)", "(b)", "(c)", "(d)"]):
    ax.add_feature(cfeature.COASTLINE)
    ax.set_extent([-80, 30, 25, 65], crs=ccrs.PlateCarree())    #Bounds: West, East, South, North

    ax.text(0.05, 0.86, char, transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground="white")])
    
    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

plt.subplots_adjust(bottom=0.08)
cbar_ax = fig.add_axes([0.35, 0.02, 0.6, 0.03])
cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")

fig.text(0.15, 0.02, "Tagged Evaporation in mm")

plt.savefig(f'./figs/figure_06.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

In [ ]:
fr_land = xr.open_dataset("data/moisture_tracking/invar/fr_land_015x015_EUL.nc")["FR_LAND"]
fr_land = fr_land.squeeze().drop_vars("time")
fr_land = fr_land.reindex(lat=list(reversed(fr_land["lat"]))).rename({"lat": "latitude", "lon": "longitude"})
fr_land = fr_land.assign_coords(latitude=fr_land["latitude"].round(6), longitude=fr_land["longitude"].round(6))

sea_regions = {}

# Mediterranean Sea:
mask_1 = ((fr_land["longitude"] >= -6) & (fr_land["longitude"] <= 37) & (fr_land["latitude"] >= 30) & (fr_land["latitude"] <= 41))
mask_2 = ((fr_land["longitude"] >= 0) & (fr_land["longitude"] <= 27) & (fr_land["latitude"] >= 40) & (fr_land["latitude"] <= 48))
sea_regions["MeS"] = {"name": "Mediterranean Sea", "mask": copy((mask_1 | mask_2) & (fr_land < 0.1))}

# Black Sea:
#mask = ((fr_land["longitude"] >= 27) & (fr_land["longitude"] <= 43) & (fr_land["latitude"] >= 40) & (fr_land["latitude"] <= 48))
#sea_regions["BlS"] = {"name": "Black Sea", "mask": copy(mask & (fr_land < 0.1))}

# Caspian Sea:
#mask = ((fr_land["longitude"] >= 46) & (fr_land["longitude"] <= 56) & (fr_land["latitude"] >= 36) & (fr_land["latitude"] <= 48))
#sea_regions["CaS"] = {"name": "Caspian Sea", "mask": copy(mask & (fr_land < 0.1))}

# Baltic Sea:
mask_1 = ((fr_land["longitude"] >= 10) & (fr_land["longitude"] <= 32) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 60))
mask_2 = ((fr_land["longitude"] >= 12) & (fr_land["longitude"] <= 29) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 70))
mask = mask_1 | mask_2
sea_regions["BaS"] = {"name": "Baltic Sea", "mask": copy((mask_1 | mask_2) & (fr_land < 0.1))}

# North Sea:
mask_1 = ((fr_land["longitude"] >= -3) & (fr_land["longitude"] <= 9) & (fr_land["latitude"] >= 53) & (fr_land["latitude"] <= 70))
mask_2 = ((fr_land["longitude"] >= -5) & (fr_land["longitude"] <= 12) & (fr_land["latitude"] >= 60) & (fr_land["latitude"] <= 70))
mask_3 = ((fr_land["longitude"] >= -5) & (fr_land["longitude"] <= 10) & (fr_land["latitude"] >= 55) & (fr_land["latitude"] <= 70))
mask = mask_1 | mask_2 | mask_3
sea_regions["NoS"] = {"name": "North Sea", "mask": copy((mask_1 | mask_2 | mask_3) & (fr_land < 0.1))}

# West European Shelf:
mask_1 = ((fr_land["longitude"] >= -3) & (fr_land["longitude"] <= 5) & (fr_land["latitude"] >= 45) & (fr_land["latitude"] <= 53))
mask_2 = ((fr_land["longitude"] >= -9) & (fr_land["longitude"] <= -1) & (fr_land["latitude"] >= 42) & (fr_land["latitude"] <= 55))
mask_3 = ((fr_land["longitude"] >= -12) & (fr_land["longitude"] <= -5) & (fr_land["latitude"] >= 50) & (fr_land["latitude"] <= 60))
mask = mask_1 | mask_2 | mask_3
sea_regions["WES"] = {"name": "West European Shelf", "mask": copy((mask_1 | mask_2 | mask_3) & (fr_land < 0.1))}

# Labrador Sea:
mask = ((fr_land["longitude"] >= -65) & (fr_land["longitude"] <= -42) & (fr_land["latitude"] >= 52) & (fr_land["latitude"] <= 65))
sea_regions["LaS"] = {"name": "Caspian Sea", "mask": copy(mask & (fr_land < 0.1))}

# North Atlantic:
mask_hudson = ((fr_land["longitude"] >= -80) & (fr_land["longitude"] <= -60) & (fr_land["latitude"] >= 50) & (fr_land["latitude"] <= 65))
mask = ((fr_land["longitude"] <= 0) & ~sea_regions["WES"]["mask"] & ~sea_regions["NoS"]["mask"] & 
        ~sea_regions["MeS"]["mask"] & ~sea_regions["LaS"]["mask"] & ~mask_hudson)
sea_regions["NAt"] = {"name": "North Atlantic", "mask": copy(mask & (fr_land < 0.1))}

del sea_regions["LaS"]

In [ ]:
area_weights = np.cos(np.deg2rad(ds_track_rea["latitude"]))
total_tracked = (area_weights * ds_track_rea["e_track"]).sum()
total_tagged  = (area_weights * ds_track_rea["tagged_precip"]).sum()
total_tracked / total_tagged * 100

In [ ]:
contributions = {}

for exp, dataset in zip(["CTL", "REA", "WLT", "SAT"], [ds_track_ctl, ds_track_rea, ds_track_wlt, ds_track_sat]):
    total = dataset["e_track"].weighted(area_weights).sum().values.item()
    unaccounted = (dataset["tagged_precip"].weighted(area_weights).sum().values.item() - total) #/ dataset["tagged_precip"].sum().values.item()

    _ = {}#{"unacc": unaccounted}

    for key in PRUDENCE_REGIONS:
        evap = dataset["e_track"].sel(latitude=PRUDENCE_REGIONS[key]["lat"], longitude=PRUDENCE_REGIONS[key]["lon"])
        evap_land = evap.where(fr_land.sel(latitude=PRUDENCE_REGIONS[key]["lat"], longitude=PRUDENCE_REGIONS[key]["lon"]).values > 0.5, other=0.)
        _[key] = evap_land.weighted(area_weights).sum().values.item() / total

    for key in sea_regions:
        _[key] = dataset["e_track"].where(sea_regions[key]["mask"], other=0.).weighted(area_weights).sum().values.item() / total

    contributions[exp] = _

In [ ]:
keys = list(contributions[list(contributions.keys())[0]].keys())
x = np.arange(len(keys))
width = 0.2

for i, (name, data) in enumerate(contributions.items()):
    values = [data[k] for k in keys]
    plt.bar(x + i*width, values, width, label=name)

plt.xticks(x + width*1.5, keys)
plt.legend()
plt.show()

In [ ]:
for key in ["REA", "CTL", "WLT", "SAT"]:  
    line = f"{key}"

    for val in contributions[key].values():
        line += f" & {100 * val:.01f}"
    
    print(line + " \\\\")

# TQV Map

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import AxesGrid
from cartopy.mpl.geoaxes import GeoAxes
import os
import contextlib

path = "/automount/agh/s6tifohr/july21_eval/data/"

with open(os.devnull, 'w') as fnull:
    with contextlib.redirect_stderr(fnull):
        CTL = xr.open_dataset(path + "BLK_CTL/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))
        SAT = xr.open_dataset(path + "BLK_SAT/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))
        WLT = xr.open_dataset(path + "BLK_WLT/2021071412/fc_R03B07_rea_ml.2021071412", engine="cfgrib", backend_kwargs=({"indexpath":""}))    

In [ ]:
colors = ['#eff3ff', '#bdd7e7', '#6baed6', '#3182bd', '#08519c', '#08306b', '#08306b', 'yellow', 'gold', 'orange', 'darkorange', 'coral']
cmap = LinearSegmentedColormap.from_list("tqv_cmap", list(zip(np.linspace(0, 1, len(colors)), colors)))
levels = np.arange(0, 60, 5)

fig = plt.figure(figsize=(10, 6))
axes_class = (GeoAxes, {"projection": ccrs.PlateCarree()})
axgr = AxesGrid(fig, 111, axes_class=axes_class, nrows_ncols=(1, 3),
                axes_pad=0.3, cbar_location="right", cbar_mode="single",
                cbar_pad=0.1, cbar_size="4%", label_mode="L")

for ax, data, title, char in zip(axgr, [CTL["TQV"], SAT["TQV"], WLT["TQV"]], ["CTL", "WET", "DRY"], ["(a)", "(b)", "(c)"]):
    cf = ax.tricontourf(grid_data["tri_26_red"], data, levels=levels, cmap=cmap, extend="neither")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="white")
    ax.set_extent([-9, 45, 36, 71], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=14)

    ax.text(0.05, 0.94, char, transform=ax.transAxes,
        ha="center", va="center",# fontweight="bold", 
        color="black", zorder=1,
        path_effects=[path_effects.withStroke(linewidth=2.5, foreground="white")])

axgr.cbar_axes[0].colorbar(cf)
axgr.cbar_axes[0].set_ylabel(r"TQV in kg m$^{-2}$", fontsize=13)

plt.savefig(f'./figs/figure_A02.png', dpi=300, bbox_inches='tight', format='png')
plt.show()